In [2]:
#library for retreiving data from dotenv file
import os
from dotenv import load_dotenv

#for model building using OpenAi
from openai import OpenAI

load_dotenv()
api_key=os.environ.get("GROQ_API_KEY")

In [3]:
from langchain_core.prompts import PromptTemplate

prompt_temp_name = PromptTemplate(
    input_variables = ['cuisine'],
    template = "I want to open a restaurant for {cuisine} food , suggest me 10 cool names. give just names not any explanation"
)

prompt_temp_name.format(cuisine = "mexican")

'I want to open a restaurant for mexican food , suggest me 10 cool names. give just names not any explanation'

In [4]:
from langchain_core.prompts import ChatPromptTemplate #for prompts 
from langchain_core.output_parsers import StrOutputParser # for structured o/p
from langchain_groq import ChatGroq #llm

#initialize basic llm using llama
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.7
)

#create prompt template
chat_prompt = ChatPromptTemplate.from_template(
    "Explain {topic} in simple terms and bullet points , not too long just ion 5 lines."
)


parser = StrOutputParser()

#this is chain using pipe operator : prompt-> llm -> o/p
chain = chat_prompt | llm | parser

response = chain.invoke({"topic" : "Generative AI"})

print(response)

Generative AI creates new content, such as:
* Images
* Text
* Music
* Videos
It uses algorithms to learn patterns and generate new data.


In [5]:
# MultiChaining -> one than one prompts layers

explain_prompt = ChatPromptTemplate.from_template(
    "Explain {topic} in detail."
)

summary_prompt = ChatPromptTemplate.from_template(
    "Summarize this text in 3 bullet points:\n\n{text}"
)

explanation_chain = explain_prompt | llm | parser

summary_chain = summary_prompt | llm | parser

#this is modern sequential chain
full_chain = explanation_chain | summary_chain

result = full_chain.invoke({"topic": "Transformers in AI"})

print(result)

Here are three bullet points summarizing the text:

* Transformers are a type of neural network architecture introduced in 2017 that revolutionized the field of natural language processing (NLP) and have since been widely adopted in various AI applications, including NLP, speech recognition, computer vision, and time series forecasting.
* The key innovation of Transformers is the use of self-attention mechanisms, which allow the model to weigh the importance of different input elements relative to each other, making them particularly well-suited for tasks that require understanding the relationships between different parts of the input sequence.
* Transformers have several advantages over traditional recurrent neural networks (RNNs), including parallelization, scalability, and improved performance, making them a powerful tool for a wide range of AI applications, and have been shown to outperform RNNs on tasks such as machine translation, text summarization, and question answering.


In [6]:
# Sequential Chain with multiple o/p
from langchain_core.runnables import RunnableParallel # it helps for multiple outputs at same time in form of dictionary

# chain = {
#     "restaurants": restaurant_chain,
#     "food_items": food_chain
# }

restaurant_prompt = ChatPromptTemplate.from_template(
    "Suggest 5 popular restaurants for {cuisine} cuisine. Give just name not any explanation."
)
restaurant_chain = restaurant_prompt | llm | parser

food_prompt = ChatPromptTemplate.from_template(
    "List 5 famous food items from {cuisine} cuisine. Give just name not any explanation."
)
food_chain = food_prompt | llm | parser

multi_output_chain = RunnableParallel(
    restaurants=restaurant_chain,
    food_items=food_chain
)
result = multi_output_chain.invoke({"cuisine": "Indian"})

print("Restaurants:\n", result["restaurants"])
print("\nFood Items:\n", result["food_items"])

Restaurants:
 1. Tandoori Nights
2. Indian Accent
3. Dhaba
4. Karavalli
5. Bukhara

Food Items:
 1. Tandoori Chicken
2. Biryani
3. Naan
4. Samosa
5. Gulab Jamun


## Tooling and Agents

In [22]:
#Creating Tools 

from langchain.tools import tool

@tool
def restaurant_finder(cuisine : str) -> str:
    """Returns popular restaurants for a given cuisine."""
    data = {
        "italian": ["Olive Garden", "La Pinoz Pizza", "Little Italy"],
        "indian": ["Bukhara", "Karim's", "Indian Accent"],
        "japanese": ["Sakura", "Megu", "Kofuku"]
    }
    return ", ".join(data.get(cuisine.lower(), ["No restaurants found"]))

@tool
def food_items(cuisine: str) -> str:
    """Returns popular food items from a cuisine."""

    data = {
        "italian": ["Pizza", "Pasta", "Risotto"],
        "indian": ["Butter Chicken", "Biryani", "Paneer Tikka"],
        "japanese": ["Sushi", "Ramen", "Tempura"]
    }

    return ", ".join(data.get(cuisine.lower(), ["No items found"]))


#registering tools
tools = [restaurant_finder, food_items]

In [23]:
#Creating an Agent

from langchain.agents import create_agent

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that recommends restaurants and food items."),
    ("human", "{input}")
])

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="You are a helpful assistant that recommends restaurants and food items."
)


response = agent.invoke({
    "messages": [{"role": "user", "content": "Suggest italian restaurants and food items"}]
})
print(response["messages"][-1].content)

I hope you find these suggestions helpful. Let me know if you need more recommendations or have any other questions.


In [24]:
for msg in response["messages"]:
    print(type(msg).__name__, ":", msg.content)

HumanMessage : Suggest italian restaurants and food items
AIMessage : 
ToolMessage : Olive Garden, La Pinoz Pizza, Little Italy
ToolMessage : Pizza, Pasta, Risotto
AIMessage : I hope you find these suggestions helpful. Let me know if you need more recommendations or have any other questions.


In [25]:
for msg in response["messages"]:
    if hasattr(msg, "tool_calls"):
        print(msg.tool_calls)

[{'name': 'restaurant_finder', 'args': {'cuisine': 'italian'}, 'id': '7bkggzd86', 'type': 'tool_call'}, {'name': 'food_items', 'args': {'cuisine': 'italian'}, 'id': 'ch79ter4c', 'type': 'tool_call'}]
[]


In [ ]:
#result in other way
result = agent.invoke({
    "messages": [{"role": "user", "content": "Suggest indian restaurants"}]
})

final_answer = result["messages"][-1].content

print(final_answer)

Some popular Indian restaurants include Bukhara, Karim's, and Indian Accent. Would you like some popular food items from these restaurants?
